# Idaho — Title 41 (Insurance) → `data/idaho/ins_codes/*.md`

The **Idaho Code** publishes **Title 41 — Insurance** on **[legislature.idaho.gov/statutesrules/idstat](https://legislature.idaho.gov/statutesrules/idstat/)**. That site is often awkward for scripted bulk export; this notebook mirrors **Justia’s** browse tree: **[Title 41 — Insurance](https://law.justia.com/codes/idaho/title-41/)** (`/codes/idaho/title-41/…`).

Each **chapter** page lists **`section-41-…`** links (for example [`…/section-41-101/`](https://law.justia.com/codes/idaho/title-41/chapter-1/section-41-101/)). **Cloudflare** commonly blocks plain **`httpx`**; we use **`curl_cffi`** with **`impersonate="chrome120"`** (same pattern as **`ins_ipynb/hawaii.ipynb`** / **`georgia.ipynb`**).

**Discovery:** BFS from the Title 41 index, following paths under **`/codes/idaho/title-41/`** that are not **`/section-…`** pages; every **`/section-…`** link under that prefix is collected (~**1,400** sections).

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`IC_sec_<label>.md`** where **`label`** is the URL tail after **`section-`** with hyphens replaced by underscores (e.g. `IC_sec_41_101.md`).

**Config:** **`CODE_YEAR`** seeds **`/codes/idaho/{year}/title-41/`** (often redirects to the same canonical tree). **`MAX_SECTIONS`** caps **downloads** (**0** = all). **`MAX_DISCOVERY_PAGES`** caps **index** fetches during discovery (**0** = no cap). **`REUSE_DISCOVERED_URLS`** skips a repeat crawl when **`_idaho_title41_section_urls.txt`** exists.

**Politeness:** **`REQUEST_DELAY_SEC`** between requests.

Then run **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import time
from collections import deque
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/idaho/title-41"
CODE_YEAR = "2024"
TITLE_INDEX = f"{BASE}/codes/idaho/{CODE_YEAR}/title-41/"

OUT_DIR = Path("data") / "idaho" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_idaho_title41_section_urls.txt"
REUSE_DISCOVERED_URLS = True


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def discover_section_urls() -> list[str]:
    """BFS Title 41 index + chapter pages; collect section URLs."""
    start = TITLE_INDEX
    seen: set[str] = set()
    in_q: set[str] = {path_key(start)}
    q: deque[str] = deque([start])
    sections: set[str] = set()
    fetches = 0
    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        pk = path_key(url)
        in_q.discard(pk)
        if pk in seen:
            continue
        if "/section-" in pk.lower():
            continue
        seen.add(pk)
        html = curl_get(url)
        fetches += 1
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(url, a["href"])
            p = path_key(absu)
            if not p.startswith(PATH_PREFIX):
                continue
            if "/section-" in p.lower():
                sections.add(BASE + p + "/")
            else:
                if p in seen or p in in_q:
                    continue
                in_q.add(p)
                q.append(BASE + p + "/")
    return sorted(sections)


def section_label_from_url(url: str) -> str:
    path = path_key(url)
    low = path.lower()
    if "/section-" not in low:
        raise ValueError(f"not a section URL: {url!r}")
    return path.rsplit("/section-", 1)[1]


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_filename(label: str) -> str:
    safe = label.replace("-", "_")
    return f"IC_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and "Idaho Code" in s:
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("ID Code §"):
            continue
        if s.startswith("Disclaimer:") or s.startswith("These codes may not"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_title41() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs under Title 41")
        all_urls = sorted(found, key=lambda u: label_sort_key(section_label_from_url(u)))
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"Idaho Code § {label}"
                md = (
                    f"# {title}\n\n"
                    f"**Idaho Code — Title 41 (Insurance)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [legislature.idaho.gov/statutesrules/idstat](https://legislature.idaho.gov/statutesrules/idstat/)\n\n"
                    f"**Section:** §{label}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_title41()


Discovered 1383 section URLs under Title 41
… 200/1383 (wrote=200 skipped=0 failed=0)
… 400/1383 (wrote=400 skipped=0 failed=0)
… 600/1383 (wrote=600 skipped=0 failed=0)
… 800/1383 (wrote=800 skipped=0 failed=0)
… 1000/1383 (wrote=1000 skipped=0 failed=0)
… 1200/1383 (wrote=1200 skipped=0 failed=0)
Done. wrote=1383 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/idaho/ins_codes


{'wrote': 1383, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
